# 対話：質問・入力・回答・確認を順に見る

授業時間は[演習カード](../activities/day1.md)を確認します。質問欄は空欄から始めます。
送信セルで `SEND=True` にした時だけ有料APIを呼びます。モデル・接続経路・許可枠を明示してください。APIキーは入力欄でのみ受け取り、保存しません。
送信失敗時は停止します。別モデルへの切替・再送は自動で行いません。大学GUIの記録とAPIの記録は経路を区別します。

In [ ]:
from pathlib import Path
import sys
PREPARE_COLAB = False  # 新しいColabで固定版を取得する時だけTrue
RELEASE_COMMIT = None  # 公開・アーカイブ検査後に固定する
RELEASE_SHA256 = None
BOOTSTRAP_SHA256 = "921585cfa741df6f607960a4f520c9ee4a551266a113ee1aa0ad102160cc43bc"
if PREPARE_COLAB:
    from urllib.request import urlopen
    from hashlib import sha256
    if not RELEASE_COMMIT or not RELEASE_SHA256:
        raise RuntimeError('この改訂版のColab公開準備はまだ完了していません。')
    url = f'https://raw.githubusercontent.com/humansys-lab/multimodal-ai-paper-reading/{RELEASE_COMMIT}/scripts/colab_bootstrap.py'
    with urlopen(url, timeout=40) as response:
        source = response.read(100_001)
    if len(source) > 100_000 or sha256(source).hexdigest() != BOOTSTRAP_SHA256:
        raise RuntimeError('準備コードのハッシュが一致しません。実行を中止しました。')
    setup = {}
    exec(compile(source, 'verified_colab_bootstrap.py', 'exec'), setup)
    ROOT = setup['prepare_notebook'](RELEASE_COMMIT, RELEASE_SHA256, install_api=True)
else:
    ROOT = globals().get('ROOT', Path.cwd()).resolve()
    if ROOT.name == 'notebooks': ROOT = ROOT.parent
    if not (ROOT / 'src/seminar_lab/__init__.py').is_file():
        raise RuntimeError('教材フォルダで実行するか、ColabではPREPARE_COLABをTrueにしてください。')
if str(ROOT / 'src') not in sys.path: sys.path.insert(0, str(ROOT / 'src'))
print('教材フォルダ:', ROOT.name)

## 1. この演習で使う道具を読み込む
画像・PDF用のライブラリはこのNotebookでは読み込みません。

In [ ]:
from seminar_lab.config import load_yaml, material_context
from seminar_lab.client import Session, LivePermission, OpenAITransport, output_text
from seminar_lab.inputs import text_input
from seminar_lab.ui import activity_view, preview_payload, save_pair
from seminar_lab.records import load_records
course = load_yaml(ROOT / 'config/course.yaml')
manifest = load_yaml(ROOT / 'materials/manifest.yaml')

## 2. 活動とモデルを明示する

教員が配布する**検証済み設定ファイル1個**を指定します。2番目のモデルを試す際も、そのモデル用の設定を明示し、新規会話にします。公開の空設定では送信できません。
`L2`＝質問の工夫、`P2a`＝共通3問の再読、`METHODS`＝追加5問、`TRANSFER`＝The AI Scientist。共通活動の論文は固定です。

In [ ]:
from uuid import uuid4
runtime_path = ROOT / 'config/runtime.example.yaml'  # 許可された非公開設定へ変更
runtime = load_yaml(runtime_path)
activity_id = 'L2'  # P2a等へ切替可。共通課題の素材は自動的にP01
print(activity_view(course, manifest, activity_id))
if 'session' not in globals():
    session = Session()
if 'run_id' not in globals():
    run_id = str(uuid4())  # このセルを再実行してもIDは維持する
question = ''
conversation_mode = 'new'  # 'continue'は成功済み会話の継続
parameters = {'max_output_tokens': 512}  # 提案値。経路の確認済み範囲以外は拒否
inputs = []

## 3. 自分で資料を抜粋する
範囲と元のページ・図・小節を書きます。質問は前のセルの `question` に記入。

In [ ]:
# P01の許可済み範囲を自分で抜粋する。質問の代わりとなる完成指示は追加しない。
source_excerpt = ''
source_location = ''
inputs = [text_input(source_excerpt, source_location)] if source_excerpt.strip() else []

## 4. 実際に送る質問と履歴を見る
少数例を加える時は質問欄に全文を書きます。隠れた指示は加えません。比較は新規会話で同じ条件にします。

In [ ]:
if question.strip():
    context = (homework_context if activity_id == 'HW1' else material_context(course, manifest, activity_id))
    print('資料と版・指定範囲:', context['material_id'], context.get('source_version'), context.get('assigned_scope'))
    print('実入力の由来:', [x.provenance for x in inputs])
    print(preview_payload(session.preview(runtime, question, inputs, conversation_mode, parameters)))
else:
    print('質問欄は空です。自分の問いを入力してください。送信していません。')

## 5. 1回だけ送信する（有料）
教員から案内されたLivePermissionを設定してください。内容・対象・上限が一致しなければ停止します。

In [ ]:
SEND = False
if 'permission' not in globals():
    permission = None  # 教員が案内した対象・上限・認証経路の許可を別セルで設定
if SEND:
    if not isinstance(permission, LivePermission):
        raise ValueError('対象・上限・認証経路を含む許可が必要です。')
    context = (homework_context if activity_id == 'HW1' else material_context(course, manifest, activity_id))
    # キー入力より先に、素材・設定・送信内容を検証
    session.preview(runtime, question, inputs, conversation_mode, parameters)
    from getpass import getpass
    transport = OpenAITransport(runtime, getpass('許可された個別APIキー（表示・保存しません）: '), permission)
    try:
        result = session.run(runtime=runtime, context=context, prompt=question, inputs=inputs,
                             mode=conversation_mode, parameters=parameters, run_id=run_id, transport=transport)
    finally:
        transport.close()
    print('状態:', result['status'])
    if result.get('error_message'):
        print(result['error_message'])
    print(output_text(result))
    print('費用:', result['cost_value'] if result['cost_value'] is not None else '不明')
else:
    print('送信OFF。APIは呼び出していません。')

## 6. 自分の説明を追記し、保存する
`None` は前の記入を保持。JSONLとMarkdownをダウンロードします。

In [ ]:
# Noneは前の記入を保持。文字列を入れた欄だけ更新する。
annotations = {
    'student_explanation': None,
    'evidence_location': None,
    'student_revision': None,
    'unresolved_point': None,
    'next_change': None,
    'change_reason': None,
}
SAVE = False
if SAVE:
    if not session.records:
        raise ValueError('保存する実行記録がありません。先に対話を実行してください。')
    if any(value is not None and not isinstance(value, str) for value in annotations.values()):
        raise ValueError('記入欄は文字列かNoneにしてください。')
    session.records[-1].update({key: value for key, value in annotations.items() if value is not None})
    paths = save_pair(session.records, ROOT / 'outputs')
    print([p.name for p in paths])
    assert load_records(paths[0]) == session.records
    assert load_records(paths[1]) == session.records
    # Colabのファイル欄からoutputs内の2ファイルをダウンロード。
else:
    print('保存は未実行です。前の記入内容も変更していません。')

## 次の質問
新しい質問の時だけ新しい実行番号を作ります。同じ送信セルの再実行による二重送信は拒否します。

In [ ]:
START_NEXT_RUN = False
if START_NEXT_RUN:
    run_id = str(uuid4())
    print('新しいrunを準備しました。質問・新規/継続・プレビューを見直してください。')

## Homeworkへの切替
先に共通課題を保存し、条件を確認した別論文と新しい会話へ切り替えます。

In [ ]:
START_HOMEWORK = False
if START_HOMEWORK:
    homework_selection = ''  # 条件を確認したE01等、またはU_<local_id>を選ぶ。P01はHW1の対象外
    local_material = None  # Uの場合は下の項目を手元で記入。空のまま送信しない
    homework_context = material_context(course, manifest, 'HW1', homework_selection, local_material)
    session = Session()  # 共通課題の保存後に新しい状態を作る
    activity_id = 'HW1'
    run_id = str(uuid4())
    conversation_mode = 'new'
    question = ''
    inputs = []
    print(activity_view(course, manifest, 'HW1'))
    print('記録段階:', homework_context['phase'])